Transformation notebook: Clean staging wind turbine data and merge into CLEANED schema with anomaly detection
*Co-authored with CoCo*

In [ ]:
%%sql -r dataframe_1
USE DATABASE WIND_TURBINE_POC;
USE SCHEMA STAGING;
USE WAREHOUSE WIND_TURBINE_POC_WH;
USE ROLE POC_ETL_ROLE;

In [ ]:
from snowflake.snowpark_connect import init_spark_session
from pyspark.sql import functions as F

spark = init_spark_session()
spark.conf.set("spark.sql.session.timeZone", "UTC")

In [ ]:
from snowflake.snowpark.context import get_active_session
from snowflake.snowpark import functions as SF
from pyspark.sql import functions as F
from pyspark.sql.types import IntegerType, DoubleType
from pyspark.sql import Window

session = get_active_session()

# 1. Get files with STATUS = 'LOADED' and their STATUS_DATETIME
loaded_files = session.table("WIND_TURBINE_POC.STAGING.FILE_LOAD_HISTORY") \
    .filter("STATUS = 'LOADED'") \
    .select("FILE_NAME", "STATUS_DATETIME") \
    .collect()

if not loaded_files:
    print("No files with STATUS 'LOADED' to process.")
else:
    file_info = {row["FILE_NAME"]: row["STATUS_DATETIME"] for row in loaded_files}
    print(f"Processing {len(file_info)} file(s): {list(file_info.keys())}")

    # 2. Read staging table and filter for LOADED files with LOADED_DATETIME >= STATUS_DATETIME
    stg_df = spark.read.table("WIND_TURBINE_POC.STAGING.STG_TURBINE_MEASUREMENT")

    file_filter = F.lit(False)
    for fname, status_dt in file_info.items():
        file_filter = file_filter | (
            (F.col("SOURCE_FILE") == F.lit(fname))
            & (F.col("LOADED_DATETIME") >= F.lit(str(status_dt)))
        )
    print (file_filter)
        
    raw_df = stg_df.filter(file_filter)
    raw_count = raw_df.count()
    print(f"Raw rows from staging: {raw_count}")

    # 3. Data type validation and conversion
    typed_df = (
        raw_df
        .withColumn("_TURBINE_ID", F.col("TURBINE_ID").cast(IntegerType()))
        .withColumn("_WIND_SPEED", F.col("WIND_SPEED").cast(DoubleType()))
        .withColumn("_WIND_DIRECTION", F.col("WIND_DIRECTION").cast(IntegerType()))
        .withColumn("_POWER_OUTPUT", F.col("POWER_OUTPUT").cast(DoubleType()))
        .withColumn("_MEASUREMENT_TS", F.to_timestamp("MEASUREMENT_TIMESTAMP"))
    )
    
    valid_df = typed_df.filter(
        F.col("_TURBINE_ID").isNotNull()
        & F.col("_WIND_SPEED").isNotNull()
        & F.col("_WIND_DIRECTION").isNotNull()
        & F.col("_POWER_OUTPUT").isNotNull()
        & F.col("_MEASUREMENT_TS").isNotNull()
    )

    print(f"filtered out by data type validation {raw_count - valid_df.count()}")

    # 4. Filter by business rules
    cleaned_df = valid_df.filter(
        (F.col("_WIND_SPEED") >= 0)
        & (F.col("_POWER_OUTPUT") >= 0)
        & (F.col("_WIND_DIRECTION") >= 0)
        & (F.col("_WIND_DIRECTION") <= 360)
    )

    print(f"filtered out by business rules {valid_df.count() - cleaned_df.count()}")

    # 5. Add MEASUREMENT_DATE and CLEANED_DATETIME
    cleaned_df = (
        cleaned_df
        .withColumn("MEASUREMENT_DATE", F.to_date(F.col("_MEASUREMENT_TS")))
        .withColumn("CLEANED_DATETIME", F.current_timestamp())
    )

    # 6. Calculate daily stats - DAILY_AVG_POWER_OUTPUT, DAILY_STDDEV_POWER_OUTPUT
    w = Window.partitionBy("_TURBINE_ID", "MEASUREMENT_DATE")
    cleaned_df = (
        cleaned_df
        .withColumn("DAILY_AVG_POWER_OUTPUT", F.round(F.avg("_POWER_OUTPUT").over(w), 3))
        .withColumn("DAILY_STDDEV_POWER_OUTPUT",
            F.coalesce(F.round(F.stddev("_POWER_OUTPUT").over(w), 3), F.lit(0.0))
        )
    )

    # 7. Calculate IS_ANOMALY: True if outside 2 standard deviations from the mean (above or below)
    cleaned_df = cleaned_df.withColumn(
        "IS_ANOMALY",
        (F.col("_POWER_OUTPUT") > (
            F.col("DAILY_AVG_POWER_OUTPUT") + 2 * F.col("DAILY_STDDEV_POWER_OUTPUT")
        ))
        | (F.col("_POWER_OUTPUT") < (
            F.col("DAILY_AVG_POWER_OUTPUT") - 2 * F.col("DAILY_STDDEV_POWER_OUTPUT")
        ))
    )

    # Get final columns
    final_df = cleaned_df.select(
        F.col("MEASUREMENT_TIMESTAMP"),
        F.col("MEASUREMENT_DATE"),
        F.col("_TURBINE_ID").alias("TURBINE_ID"),
        F.round(F.col("_WIND_SPEED"), 3).alias("WIND_SPEED"),
        F.col("_WIND_DIRECTION").alias("WIND_DIRECTION"),
        F.round(F.col("_POWER_OUTPUT"), 3).alias("POWER_OUTPUT"),
        F.col("DAILY_AVG_POWER_OUTPUT"),
        F.col("DAILY_STDDEV_POWER_OUTPUT"),
        F.col("IS_ANOMALY"),
        F.col("SOURCE_FILE"),
        F.col("LOADED_DATETIME"),
        F.col("CLEANED_DATETIME")
    )

    cleaned_count = final_df.count()
    print(f"Cleaned rows: {cleaned_count} (filtered out {raw_count - cleaned_count})")

    # Save to temp table for SQL operations 
    final_df.write.mode("overwrite").saveAsTable(
        "WIND_TURBINE_POC.STAGING.CLEANED_MEASUREMENTS_TEMP"
    )

    # 8. INSERT new turbines into CLEANED.TURBINE
    insert_result = session.sql("""
        INSERT INTO WIND_TURBINE_POC.CLEANED.TURBINE (TURBINE_ID, SOURCE_FILE, CREATED_DATETIME)
        SELECT  TURBINE_ID, SOURCE_FILE, CREATED_DATETIME
        FROM 
        (  SELECT  TURBINE_ID, 
                    MAX(SOURCE_FILE) as SOURCE_FILE,
                    MAX(CLEANED_DATETIME) AS CREATED_DATETIME
            FROM WIND_TURBINE_POC.STAGING.CLEANED_MEASUREMENTS_TEMP 
            GROUP BY TURBINE_ID
        ) nt
        WHERE NOT EXISTS (
            SELECT 1 FROM WIND_TURBINE_POC.CLEANED.TURBINE t
            WHERE t.TURBINE_ID = nt.TURBINE_ID )
        
    """).collect()
    print(f"Turbine insert result: {insert_result}")

    # 9. MERGE into CLEANED.TURBINE_MEASUREMENT (PK: MEASUREMENT_TIMESTAMP, TURBINE_ID)
    merge_result = session.sql("""
        MERGE INTO WIND_TURBINE_POC.CLEANED.TURBINE_MEASUREMENT AS target
        USING WIND_TURBINE_POC.STAGING.CLEANED_MEASUREMENTS_TEMP AS source
        ON target.MEASUREMENT_TIMESTAMP = source.MEASUREMENT_TIMESTAMP
           AND target.TURBINE_ID = source.TURBINE_ID
        WHEN MATCHED THEN UPDATE SET
            target.MEASUREMENT_DATE           = source.MEASUREMENT_DATE,
            target.WIND_SPEED                 = source.WIND_SPEED,
            target.WIND_DIRECTION             = source.WIND_DIRECTION,
            target.POWER_OUTPUT               = source.POWER_OUTPUT,
            target.DAILY_AVG_POWER_OUTPUT     = source.DAILY_AVG_POWER_OUTPUT,
            target.DAILY_STDDEV_POWER_OUTPUT  = source.DAILY_STDDEV_POWER_OUTPUT,
            target.IS_ANOMALY                 = source.IS_ANOMALY,
            target.SOURCE_FILE                = source.SOURCE_FILE,
            target.LOADED_DATETIME            = source.LOADED_DATETIME,
            target.CLEANED_DATETIME           = source.CLEANED_DATETIME
        WHEN NOT MATCHED THEN INSERT (
            MEASUREMENT_TIMESTAMP, MEASUREMENT_DATE, TURBINE_ID,
            WIND_SPEED, WIND_DIRECTION, POWER_OUTPUT,
            DAILY_AVG_POWER_OUTPUT, DAILY_STDDEV_POWER_OUTPUT, IS_ANOMALY,
            SOURCE_FILE, LOADED_DATETIME, CLEANED_DATETIME
        ) VALUES (
            source.MEASUREMENT_TIMESTAMP, 
            source.MEASUREMENT_DATE, 
            source.TURBINE_ID,
            source.WIND_SPEED, 
            source.WIND_DIRECTION, 
            source.POWER_OUTPUT,
            source.DAILY_AVG_POWER_OUTPUT, 
            source.DAILY_STDDEV_POWER_OUTPUT, 
            source.IS_ANOMALY,
            source.SOURCE_FILE, 
            source.LOADED_DATETIME, 
            source.CLEANED_DATETIME
        )
    """).collect()
    print(f"Merge result: {merge_result}")

    session.table("WIND_TURBINE_POC.CLEANED.TURBINE_MEASUREMENT").limit(5).show()

In [ ]:
# Populate Stats Mart if temp table exists
table_exists = session.sql("""
    SELECT COUNT(*) AS CNT
    FROM WIND_TURBINE_POC.INFORMATION_SCHEMA.TABLES
    WHERE TABLE_SCHEMA = 'STAGING'
      AND TABLE_NAME = 'CLEANED_MEASUREMENTS_TEMP'
""").collect()[0]["CNT"] > 0

if not table_exists:
    print("There is no data to merge - STAGING.CLEANED_MEASUREMENTS_TEMP does not exist. Skipping stats mart merge.")
else:
    merge_result = session.sql("""
        MERGE INTO WIND_TURBINE_POC.MART.DAILY_TURBINE_STATS AS target
        USING ( SELECT MEASUREMENT_DATE,
                       TURBINE_ID,
                       COUNT(1) AS MEASUREMENT_COUNT,
                       MIN(POWER_OUTPUT) AS MIN_POWER_OUTPUT,
                       MAX(POWER_OUTPUT) AS MAX_POWER_OUTPUT,
                       AVG(POWER_OUTPUT) AS AVG_POWER_OUTPUT,
                       MAX(CLEANED_DATETIME) AS AGGREGATION_DATETIME
                FROM WIND_TURBINE_POC.STAGING.CLEANED_MEASUREMENTS_TEMP
                GROUP BY MEASUREMENT_DATE, TURBINE_ID
            ) AS source
        ON target.MEASUREMENT_DATE = source.MEASUREMENT_DATE
           AND target.TURBINE_ID = source.TURBINE_ID
        WHEN MATCHED THEN UPDATE SET
            target.MEASUREMENT_COUNT    = source.MEASUREMENT_COUNT,
            target.MIN_POWER_OUTPUT     = source.MIN_POWER_OUTPUT,
            target.MAX_POWER_OUTPUT     = source.MAX_POWER_OUTPUT,
            target.AVG_POWER_OUTPUT     = source.AVG_POWER_OUTPUT,
            target.AGGREGATION_DATETIME = source.AGGREGATION_DATETIME
        WHEN NOT MATCHED THEN INSERT (
            MEASUREMENT_DATE, TURBINE_ID, MEASUREMENT_COUNT,
            MIN_POWER_OUTPUT, MAX_POWER_OUTPUT, AVG_POWER_OUTPUT,
            AGGREGATION_DATETIME
        ) VALUES (
            source.MEASUREMENT_DATE,
            source.TURBINE_ID,
            source.MEASUREMENT_COUNT,
            source.MIN_POWER_OUTPUT,
            source.MAX_POWER_OUTPUT,
            source.AVG_POWER_OUTPUT,
            source.AGGREGATION_DATETIME
        )
    """).collect()
    print(f"Stats mart merge result: {merge_result}")

In [ ]:
%%sql -r dataframe_3
UPDATE WIND_TURBINE_POC.STAGING.FILE_LOAD_HISTORY
SET STATUS = 'COMPLETED',
    STATUS_DATETIME = SYSDATE()
WHERE STATUS = 'LOADED';

In [ ]:
# Cleanup temp table
session.sql("DROP TABLE IF EXISTS WIND_TURBINE_POC.STAGING.CLEANED_MEASUREMENTS_TEMP").collect()